In [0]:
# ecommerce_order_fulfilment
# Name: Raj Kumar
# Dataset: brazilian-ecommerce

### Setup `Catalog`

In [0]:
#  Databricks version
import os
print(os.environ.get("DATABRICKS_RUNTIME_VERSION"))

client.5.10


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecom;
USE CATALOG ecom;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecom.gold;
CREATE SCHEMA IF NOT EXISTS ecom.silver;
CREATE SCHEMA IF NOT EXISTS ecom.bronze;

### utilities

In [0]:
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

### import libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.functions import to_date
from pyspark.sql.functions import month
from pyspark.sql.functions import round
from pyspark.sql.functions import when

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "ecom", "catalog")
dbutils.widgets.text("data_source1", "customers", "data_source1")
dbutils.widgets.text("data_source2", "geolocation", "data_source2")
dbutils.widgets.text("data_source3", "order_items", "data_source3")
dbutils.widgets.text("data_source4", "order_payments", "data_source4")
dbutils.widgets.text("data_source5", "order_reviews", "data_source5")
dbutils.widgets.text("data_source6", "orders", "data_source6")
dbutils.widgets.text("data_source7", "products", "data_source7")
dbutils.widgets.text("data_source8", "sellers", "data_source8")
dbutils.widgets.text("data_source9", "product_category_name_translation", "data_source9")

catalog = dbutils.widgets.get("catalog")
data_source1 = dbutils.widgets.get("data_source1")
data_source2 = dbutils.widgets.get("data_source2")
data_source3 = dbutils.widgets.get("data_source3")
data_source4 = dbutils.widgets.get("data_source4")
data_source5 = dbutils.widgets.get("data_source5")
data_source6 = dbutils.widgets.get("data_source6")
data_source7 = dbutils.widgets.get("data_source7")
data_source8 = dbutils.widgets.get("data_source8")
data_source9 = dbutils.widgets.get("data_source9")

print(catalog, data_source1, data_source2, data_source3, data_source4, data_source5, data_source6, data_source7, data_source8, data_source9)

ecom customers geolocation order_items order_payments order_reviews orders products sellers product_category_name_translation


# Load Dataset in Databricks
### Read Dataset Using Learner-Written Code

### 1. Customers

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_customers_dataset.csv"
# create DataFrame
df_customers = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_customers.limit(10))

# print check data type
df_customers.printSchema()
print(f'total row count: {df_customers.count()}')

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

total row count: 99441


### 2. Geolocation

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_geolocation_dataset.csv"
# create DataFrame
df_geolocation = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_geolocation.limit(10))

# print check data type
df_geolocation.printSchema()
print(f'total row count: {df_geolocation.count()}')

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP
1012,-23.547762303364266,-46.63536053788448,são paulo,SP
1047,-23.546273112412678,-46.64122516971552,sao paulo,SP
1013,-23.546923208436723,-46.6342636964915,sao paulo,SP
1029,-23.543769055769133,-46.63427784085132,sao paulo,SP
1011,-23.547639550320632,-46.63603162315495,sao paulo,SP


root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

total row count: 1000163


### 3. order_items

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_order_items_dataset.csv"
# create DataFrame
df_order_items = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_order_items.limit(10))

# print check data type
df_order_items.printSchema()
print(f'total row count: {df_order_items.count()}')

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

total row count: 112650


### 4. order_payments

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_order_payments_dataset.csv"
# create DataFrame
df_order_payments = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_order_payments.limit(10))

# print check data type
df_order_payments.printSchema()
print(f'total row count: {df_order_payments.count()}')

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

total row count: 103886


### 5. order_reviews

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_order_reviews_dataset.csv"
# create DataFrame
df_order_reviews = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_order_reviews.limit(10))

# print check data type
df_order_reviews.printSchema()
print(f'total row count: {df_order_reviews.count()}')

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)

total row count: 104162


#### 6. orders

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_orders_dataset.csv"
# create DataFrame
df_orders = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_orders.limit(10))

# print check data type
df_orders.printSchema()
print(f'total row count: {df_orders.count()}')

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,02-10-2017 10:56,02-10-2017 11:07,04-10-2017 19:55,10-10-2017 21:25,18-10-2017 00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,24-07-2018 20:41,26-07-2018 03:24,26-07-2018 14:31,07-08-2018 15:27,13-08-2018 00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08-08-2018 08:38,08-08-2018 08:55,08-08-2018 13:50,17-08-2018 18:06,04-09-2018 00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,18-11-2017 19:28,18-11-2017 19:45,22-11-2017 13:39,02-12-2017 00:28,15-12-2017 00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,13-02-2018 21:18,13-02-2018 22:20,14-02-2018 19:46,16-02-2018 18:17,26-02-2018 00:00
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,09-07-2017 21:57,09-07-2017 22:10,11-07-2017 14:58,26-07-2017 10:57,01-08-2017 00:00
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,11-04-2017 12:22,13-04-2017 13:25,null,null,09-05-2017 00:00
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,16-05-2017 13:10,16-05-2017 13:22,22-05-2017 10:07,26-05-2017 12:55,07-06-2017 00:00
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,23-01-2017 18:29,25-01-2017 02:50,26-01-2017 14:16,02-02-2017 14:08,06-03-2017 00:00
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,29-07-2017 11:55,29-07-2017 12:05,10-08-2017 19:45,16-08-2017 17:14,23-08-2017 00:00


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

total row count: 99441


### 7. products

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_products_dataset.csv"
# create DataFrame
df_products = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_products.limit(10))

# print check data type
df_products.printSchema()
print(f'total row count: {df_products.count()}')

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

total row count: 32951


### 8. sellers

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/olist_sellers_dataset.csv"
# create DataFrame
df_sellers = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_sellers.limit(10))

# print check data type
df_sellers.printSchema()
print(f'total row count: {df_sellers.count()}')

seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

total row count: 3095


### 9. product_category_name_translation

In [0]:
base_path = "s3://ecommerce-order-fulfilment/raw/product_category_name_translation.csv"
# create DataFrame
df_product_category_name_translation = (spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load(base_path)
                )

display(df_product_category_name_translation.limit(10))

# print check data type
df_product_category_name_translation.printSchema()
print(f'total row count: {df_product_category_name_translation.count()}')

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

total row count: 71


# Create Bronze Delta Table
### Create Bronze Delta Table Using Learner-Written Code

In [0]:
# write data in bronze layer
df_customers.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source1}')

df_geolocation.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source2}')

df_order_items.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source3}')

df_order_payments.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source4}')

df_order_reviews.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source5}')

df_orders.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source6}')

df_products.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source7}')

df_sellers.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source8}')

df_product_category_name_translation.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", True)\
            .mode("overwrite")\
                .saveAsTable(f'{catalog}.{bronze_schema}.{data_source9}')

# SILVER LAYER GAME PLAN
In Silver, we will:

✅ Clean data
✅ Fix data types
✅ Handle nulls
✅ Deduplicate
✅ Standardize columns
✅ Basic validations

### STEP 1 — Read Bronze Tables

In [0]:
orders_df = spark.table("ecom.bronze.orders")
customers_df = spark.table("ecom.bronze.customers")
order_items_df = spark.table("ecom.bronze.order_items")
payments_df = spark.table("ecom.bronze.order_payments")
reviews_df = spark.table("ecom.bronze.order_reviews")
products_df = spark.table("ecom.bronze.products")
sellers_df = spark.table("ecom.bronze.sellers")
geo_df = spark.table("ecom.bronze.geolocation")
category_df = spark.table("ecom.bronze.product_category_name_translation")

In [0]:
orders_df.printSchema()
display(orders_df.limit(10))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,02-10-2017 10:56,02-10-2017 11:07,04-10-2017 19:55,10-10-2017 21:25,18-10-2017 00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,24-07-2018 20:41,26-07-2018 03:24,26-07-2018 14:31,07-08-2018 15:27,13-08-2018 00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08-08-2018 08:38,08-08-2018 08:55,08-08-2018 13:50,17-08-2018 18:06,04-09-2018 00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,18-11-2017 19:28,18-11-2017 19:45,22-11-2017 13:39,02-12-2017 00:28,15-12-2017 00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,13-02-2018 21:18,13-02-2018 22:20,14-02-2018 19:46,16-02-2018 18:17,26-02-2018 00:00
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,09-07-2017 21:57,09-07-2017 22:10,11-07-2017 14:58,26-07-2017 10:57,01-08-2017 00:00
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,11-04-2017 12:22,13-04-2017 13:25,null,null,09-05-2017 00:00
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,16-05-2017 13:10,16-05-2017 13:22,22-05-2017 10:07,26-05-2017 12:55,07-06-2017 00:00
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,23-01-2017 18:29,25-01-2017 02:50,26-01-2017 14:16,02-02-2017 14:08,06-03-2017 00:00
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,29-07-2017 11:55,29-07-2017 12:05,10-08-2017 19:45,16-08-2017 17:14,23-08-2017 00:00


### STEP 2 — Clean ORDERS Table
- **Transformations**:
- Remove null order_id
- Convert timestamps
- Remove duplicates

In [0]:
from pyspark.sql.functions import col, to_timestamp

orders_clean = (
    orders_df
    .filter(col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_ts", to_timestamp("order_purchase_timestamp", "dd-MM-yyyy HH:mm"))
    .withColumn("order_approved_ts", to_timestamp("order_approved_at", "dd-MM-yyyy HH:mm"))
    .withColumn("order_delivered_ts", to_timestamp("order_delivered_customer_date", "dd-MM-yyyy HH:mm"))
    .withColumn("order_estimated_delivery_ts", to_timestamp("order_estimated_delivery_date", "dd-MM-yyyy HH:mm"))
    .drop(
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

### Write to Silver

In [0]:
orders_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.orders")

### STEP 3 — Clean CUSTOMERS Table
#### Transformations:
- Remove nulls
- Standardize city/state
- Deduplicate


In [0]:
from pyspark.sql.functions import upper, trim

customers_clean = (
    customers_df
    .filter(col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", upper(trim(col("customer_city"))))
    .withColumn("customer_state", upper(col("customer_state")))
)

customers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.customers")

### STEP 4 — Clean ORDER ITEMS
#### Transformations:
- Remove invalid price
- Cast numeric columns
- Deduplicate

In [0]:
from pyspark.sql.functions import col

order_items_clean = (
    order_items_df
    .filter(col("order_id").isNotNull())
    .filter(col("price") > 0)
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
)

order_items_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.order_items")

### STEP 5 — Clean PAYMENTS
#### Transformations:
- Remove null order_id
- Cast payment value

In [0]:
payments_clean = (
    payments_df
    .filter(col("order_id").isNotNull())
    .withColumn("payment_value", col("payment_value").cast("double"))
)

payments_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.order_payments")

### STEP 6 — Clean REVIEWS
#### Transformations:
- Remove null review_id
- Cast review score

In [0]:


reviews_clean = (
    reviews_df_from_source
    .filter(col("review_id").isNotNull())
    .dropDuplicates(["review_id"])
    .withColumn("review_score", expr("try_cast(review_score as int)"))
)

reviews_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.order_reviews")

### STEP 7 — Clean PRODUCTS

In [0]:
products_clean = (
    products_df
    .filter(col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

products_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.products")

### STEP 8 — Clean SELLERS

In [0]:
sellers_clean = (
    sellers_df
    .filter(col("seller_id").isNotNull())
    .dropDuplicates(["seller_id"])
)

sellers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.sellers")

### STEP 9 — Clean GEOLOCATION

In [0]:
geo_clean = (
    geo_df
    .withColumn("geolocation_city", upper(trim(col("geolocation_city"))))
)

geo_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.geolocation")

### STEP 10 — Category Translation

In [0]:
category_clean = (
    category_df
    .dropDuplicates(["product_category_name"])
)

category_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.category_translation")

### STEP 1 — Start from Clean Silver Orders

In [0]:
from pyspark.sql.functions import col, datediff

orders_df = spark.table("ecom.silver.orders")

### STEP 2 — Delivery Delay Calculation
#### Business Logic:
- Delivery time = delivered_ts - purchase_ts
- If not delivered → NULL

In [0]:
orders_enriched = (
    orders_df
    .withColumn(
        "delivery_days",
        datediff(col("order_delivered_ts"), col("order_purchase_ts"))
    )
)

### STEP 3 — Estimated vs Actual Delivery (Key KPI)
#### Business Logic:
- Compare actual vs estimated delivery

In [0]:
orders_enriched = (
    orders_enriched
    .withColumn(
        "estimated_delivery_days",
        datediff(col("order_estimated_delivery_ts"), col("order_purchase_ts"))
    )
)

### STEP 4 — SLA Breach Flag 
#### Business Logic:
- If delivered after estimated date → SLA breach

In [0]:
from pyspark.sql.functions import when

orders_enriched = (
    orders_enriched
    .withColumn(
        "is_sla_breached",
        when(
            col("order_delivered_ts") > col("order_estimated_delivery_ts"),
            1
        ).otherwise(0)
    )
)

### STEP 5 — Handle Edge Cases (Production Level)
#### Important Scenarios:
- Not delivered yet
- Missing timestamps

In [0]:
orders_enriched = (
    orders_enriched
    .withColumn(
        "is_sla_breached",
        when(col("order_delivered_ts").isNull(), None)
        .when(col("order_estimated_delivery_ts").isNull(), None)
        .when(col("order_delivered_ts") > col("order_estimated_delivery_ts"), 1)
        .otherwise(0)
    )
)

### STEP 6 — Delivery Performance Category
#### Business Classification:
- Early
- On-time
- Delayed

In [0]:
orders_enriched = (
    orders_enriched
    .withColumn(
        "delivery_status",
        when(col("order_delivered_ts").isNull(), "NOT_DELIVERED")
        .when(col("order_delivered_ts") < col("order_estimated_delivery_ts"), "EARLY")
        .when(col("order_delivered_ts") == col("order_estimated_delivery_ts"), "ON_TIME")
        .otherwise("DELAYED")
    )
)

### STEP 7 — Delay Severity Buckets ( Advanced Feature)

In [0]:
orders_enriched = (
    orders_enriched
    .withColumn(
        "delay_days",
        datediff(col("order_delivered_ts"), col("order_estimated_delivery_ts"))
    )
    .withColumn(
        "delay_bucket",
        when(col("delay_days") <= 0, "NO_DELAY")
        .when(col("delay_days") <= 3, "MINOR_DELAY")
        .when(col("delay_days") <= 7, "MODERATE_DELAY")
        .otherwise("SEVERE_DELAY")
    )
)

### STEP 8 — Final Silver Orders Table

In [0]:
orders_enriched.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.orders_enriched")

### STEP 9 — Data Quality Checks (MUST HAVE)

In [0]:
# No negative delivery days
orders_enriched.filter(col("delivery_days") < 0).count()

# SLA breach sanity
orders_enriched.groupBy("is_sla_breached").count().show()

# Delay distribution
orders_enriched.groupBy("delay_bucket").count().show()

+---------------+-----+
|is_sla_breached|count|
+---------------+-----+
|              0|88649|
|              1| 7827|
|           NULL| 2965|
+---------------+-----+

+--------------+-----+
|  delay_bucket|count|
+--------------+-----+
|      NO_DELAY|89941|
|  SEVERE_DELAY| 5828|
|   MINOR_DELAY| 1870|
|MODERATE_DELAY| 1802|
+--------------+-----+



### BUSINESS METRICS FOR SELLERS

#### We will calculate:

#### Core KPIs
- Total orders per seller
- Total revenue
- Avg delivery time
- SLA breach rate
#### Advanced KPIs
- Delay rate
- Avg delay days
- Seller performance category

# STEP 1 — Load Required Table

In [0]:
from pyspark.sql.functions import col

orders_df = spark.table("ecom.silver.orders_enriched")
order_items_df = spark.table("ecom.silver.order_items")
sellers_df = spark.table("ecom.silver.sellers")

### STEP 2 — Join Orders with Order Items

In [0]:
orders_items_joined = (
    order_items_df
    .join(orders_df, on="order_id", how="inner")
)

### STEP 3 — Add Seller Info

In [0]:
seller_orders = (
    orders_items_joined
    .join(sellers_df, on="seller_id", how="left")
)

### STEP 4 — Compute Seller-Level Aggregations

In [0]:
from pyspark.sql.functions import (
    countDistinct, sum, avg, when
)

seller_performance = (
    seller_orders
    .groupBy("seller_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("price").alias("total_revenue"),
        avg("delivery_days").alias("avg_delivery_days"),
        avg("delay_days").alias("avg_delay_days"),
        sum(when(col("is_sla_breached") == 1, 1).otherwise(0)).alias("sla_breach_count"),
        countDistinct("order_id").alias("total_orders_check")
    )
)

### STEP 5 — SLA Breach Rate

In [0]:
from pyspark.sql.functions import expr

seller_performance = (
    seller_performance
    .withColumn(
        "sla_breach_rate",
        col("sla_breach_count") / col("total_orders")
    )
)

### STEP 6 — Seller Performance Classification 

In [0]:
from pyspark.sql.functions import when

seller_performance = (
    seller_performance
    .withColumn(
        "seller_performance_category",
        when(col("sla_breach_rate") < 0.05, "EXCELLENT")
        .when(col("sla_breach_rate") < 0.15, "GOOD")
        .when(col("sla_breach_rate") < 0.30, "AVERAGE")
        .otherwise("POOR")
    )
)

### STEP 7 — Join Seller Location Info

In [0]:
seller_performance = (
    seller_performance
    .join(
        sellers_df.select("seller_id", "seller_city", "seller_state"),
        on="seller_id",
        how="left"
    )
)

### STEP 8 — Write to Silver Layer

In [0]:
seller_performance.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.seller_performance")

### STEP 9 — Data Validation (VERY IMPORTANT)

In [0]:
# Check distribution
seller_performance.groupBy("seller_performance_category").count().show()

# Check top sellers
seller_performance.orderBy(col("total_revenue").desc()).show(10)

# Check SLA issues
seller_performance.filter(col("sla_breach_rate") > 0.3).show()

+---------------------------+-----+
|seller_performance_category|count|
+---------------------------+-----+
|                    AVERAGE|  300|
|                  EXCELLENT| 1921|
|                       GOOD|  645|
|                       POOR|  229|
+---------------------------+-----+

+--------------------+------------+------------------+------------------+-------------------+----------------+------------------+-------------------+---------------------------+----------------+------------+
|           seller_id|total_orders|     total_revenue| avg_delivery_days|     avg_delay_days|sla_breach_count|total_orders_check|    sla_breach_rate|seller_performance_category|     seller_city|seller_state|
+--------------------+------------+------------------+------------------+-------------------+----------------+------------------+-------------------+---------------------------+----------------+------------+
|4869f7a5dfa277a7d...|        1132|229472.62999999808|14.936411149825783|-11.1646341463

### Seller + Region Performance

In [0]:
from pyspark.sql.functions import col, avg, sum, when, countDistinct

seller_orders = (
    spark.table("ecom.silver.order_items")
    .join(spark.table("ecom.silver.orders_enriched"), "order_id")
    .join(spark.table("ecom.silver.sellers"), "seller_id")
)

seller_region_perf = (
    seller_orders
    .groupBy("seller_id", "seller_state")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        avg("delivery_days").alias("avg_delivery_days"),
        avg("delay_days").alias("avg_delay_days"),
        sum(when(col("is_sla_breached") == 1, 1).otherwise(0)).alias("sla_breaches")
    )
    .withColumn(
        "sla_breach_rate",
        col("sla_breaches") / col("total_orders")
    )
)

seller_region_perf.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.seller_region_performance")

### Revenue Analysis

In [0]:
revenue_df = (
    spark.table("ecom.silver.order_items")
    .join(spark.table("ecom.silver.order_payments"), "order_id")
    .join(spark.table("ecom.silver.products"), "product_id")
    .join(spark.table("ecom.silver.category_translation"), "product_category_name")
)

revenue_analysis = (
    revenue_df
    .groupBy("payment_type", "product_category_name_english")
    .agg(
        avg("price").alias("avg_price"),
        avg("freight_value").alias("avg_freight"),
        sum("payment_value").alias("total_revenue")
    )
)

revenue_analysis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.revenue_analysis")

### Review vs Delivery Delay

In [0]:
from pyspark.sql.functions import countDistinct

review_df = (
    spark.table("ecom.silver.orders_enriched")
    .join(spark.table("ecom.silver.order_reviews"), "order_id")
)

review_analysis = (
    review_df
    .groupBy("delay_bucket")
    .agg(
        avg("review_score").alias("avg_review_score"),
        countDistinct("order_id").alias("total_orders")
    )
)

review_analysis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.review_delay_analysis")

### Category Performance (Revenue vs Experience)

In [0]:
category_perf = (
    spark.table("ecom.silver.order_items")
    .join(spark.table("ecom.silver.products"), "product_id")
    .join(spark.table("ecom.silver.category_translation"), "product_category_name")
    .join(spark.table("ecom.silver.order_reviews"), "order_id")
    .groupBy("product_category_name_english")
    .agg(
        sum("price").alias("total_revenue"),
        avg("review_score").alias("avg_review_score")
    )
)

category_perf.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.silver.category_performance")

### GOLD LAYER OBJECTIVE

#### We’re transforming Silver data into:

- Business-ready
- Aggregated
- Fast for dashboards
- Star schema (Fact + Dimensions)

### STEP 1: DESIGN STAR SCHEMA
#### Fact Tables (metrics)
- fact_orders
- fact_payments
#### Dimension Tables (descriptive)
- dim_customers
- dim_sellers
- dim_products
- dim_date

### STEP 2: DIMENSION TABLES

#### 1. Customer Dimension

In [0]:
dim_customers = (
    spark.table("ecom.silver.customers")
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates()
)

dim_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.dim_customers")

#### 2. Seller Dimension

In [0]:
dim_sellers = (
    spark.table("ecom.silver.sellers")
    .select(
        "seller_id",
        "seller_city",
        "seller_state"
    )
    .dropDuplicates()
)

dim_sellers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.dim_sellers")

#### 3. Product Dimension

In [0]:
dim_products = (
    spark.table("ecom.silver.products")
    .join(
        spark.table("ecom.silver.category_translation"),
        "product_category_name",
        "left"
    )
    .select(
        "product_id",
        "product_category_name_english"
    )
    .dropDuplicates()
)

dim_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.dim_products")

#### 4. Date Dimension (VERY IMPORTANT)

In [0]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, weekofyear

dim_date = (
    spark.table("ecom.silver.orders_enriched")
    .select(
        to_date("order_purchase_ts").alias("date")
    )
    .dropDuplicates()
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("week", weekofyear("date"))
)

dim_date.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.dim_date")

### STEP 3: FACT TABLES

#### 1. Fact Orders (CORE TABLE)

In [0]:
from pyspark.sql.functions import col

fact_orders = (
    spark.table("ecom.silver.orders_enriched")
    .join(spark.table("ecom.silver.order_items"), "order_id")
    .join(spark.table("ecom.silver.order_reviews"), "order_id", "left")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "seller_id",
        col("order_purchase_ts").alias("order_date"),
        "price",
        "freight_value",
        "delivery_days",
        "delay_days",
        "is_sla_breached",
        "review_score"
    )
)

fact_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.fact_orders")

#### 2. Fact Payments

In [0]:
fact_payments = (
    spark.table("ecom.silver.order_payments")
    .select(
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
)

fact_payments.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.fact_payments")

### STEP 4: STAR SCHEMA RELATIONSHIP

![image_1785674744856.png](./image_1785674744856.png "image_1785674744856.png")

### STEP 5: BUSINESS MARTS

#### Seller Performance Mart

In [0]:
from pyspark.sql.functions import avg, sum

seller_perf_mart = (
    spark.table("ecom.gold.fact_orders")
    .groupBy("seller_id")
    .agg(
        avg("delivery_days").alias("avg_delivery"),
        avg("delay_days").alias("avg_delay"),
        avg("review_score").alias("avg_review"),
        sum("is_sla_breached").alias("total_sla_breaches")
    )
)

seller_perf_mart.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecom.gold.mart_seller_performance")

### Final Architecture

![image_1785675064026.png](./image_1785675064026.png "image_1785675064026.png")

### 1. Sellers with Highest Delay Rate

In [0]:
%sql
SELECT 
    seller_id,
    COUNT(*) AS total_orders,
    SUM(is_sla_breached) AS delayed_orders,
    ROUND(SUM(is_sla_breached)/COUNT(*), 3) AS delay_rate
FROM ecom.gold.fact_orders
GROUP BY seller_id
HAVING COUNT(*) > 50   -- avoid small sample bias
ORDER BY delay_rate DESC
LIMIT 10;

seller_id,total_orders,delayed_orders,delay_rate
54965bbe3e4f07ae045b90b0b8541f52,86,26,0.302
2a1348e9addc1af5aaa619b1a3679d6b,55,15,0.273
6039e27294dc75811c0d8a39069f52c0,75,19,0.253
602044f2c16190c2c6e45eb35c2e21cb,60,15,0.25
a49928bcdf77c55c6d6e05e09a9b4ca5,106,26,0.245
beadbee30901a7f61d031b6b686095ad,68,16,0.235
06a2c3af7b3aee5d69171b0e14f0ee87,406,95,0.234
ea566164622c6b439516ab18062c42cd,52,12,0.231
cac4c8e7b1ca6252d8f20b2fc1a2e4af,83,19,0.229
bbad7e518d7af88a0897397ffdca1979,85,19,0.224



### Interpretation
#### Top sellers here are your worst performers
- Focus especially on:
- High delay rate (>30%)
- AND high order volume

#### These sellers are:

- Hurting customer experience
- Likely causing bad reviews

### 2. Regions with Highest Delay Rate

In [0]:
%sql
SELECT 
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(f.is_sla_breached) AS delayed_orders,
    ROUND(SUM(f.is_sla_breached)/COUNT(*), 3) AS delay_rate
FROM ecom.gold.fact_orders f
JOIN ecom.gold.dim_customers c ON f.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY delay_rate DESC;

customer_state,total_orders,delayed_orders,delay_rate
AL,447,104,0.233
MA,826,163,0.197
SE,385,61,0.158
PI,543,81,0.149
CE,1480,218,0.147
BA,3807,505,0.133
RJ,14622,1841,0.126
TO,315,38,0.121
PA,1083,131,0.121
ES,2263,273,0.121


### Expected Pattern (from your dataset behavior)
#### High Delay Regions
- Remote states / low logistics coverage
- Long-distance shipping zones
#### Medium Delay
- Semi-urban areas
#### Low Delay
- Metro cities (better infrastructure)

### Which sellers or regions have the highest delivery delay rate?

In [0]:
%sql
SELECT 
    f.seller_id,
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(f.is_sla_breached) AS delayed_orders,
    ROUND(SUM(f.is_sla_breached)/COUNT(*), 3) AS delay_rate
FROM ecom.gold.fact_orders f
JOIN ecom.gold.dim_customers c ON f.customer_id = c.customer_id
GROUP BY f.seller_id, c.customer_state
HAVING COUNT(*) > 30
ORDER BY delay_rate DESC;

seller_id,customer_state,total_orders,delayed_orders,delay_rate
2709af9587499e95e803a6498a5a56e9,SP,31,21,0.677
88460e8ebdecbfecb5f9601833981930,MG,36,18,0.5
7aa4334be125fcdd2ba64b3180029f14,RJ,35,14,0.4
cac4c8e7b1ca6252d8f20b2fc1a2e4af,SP,31,10,0.323
8160255418d5aaa7dbdc9f4c64ebda44,RJ,76,23,0.303
d13e50eaa47b4cbe9eb81465865d8cfc,SP,48,14,0.292
c826c40d7b19f62a09e2d7c5e7295ee2,SC,31,9,0.29
88460e8ebdecbfecb5f9601833981930,RJ,45,13,0.289
897060da8b9a21f655304d50fd935913,RJ,56,16,0.286
06a2c3af7b3aee5d69171b0e14f0ee87,SP,131,35,0.267


### Final Business Insight

- A small group of sellers contributes disproportionately to delivery delays
- Delivery delays are higher in remote regions due to logistics constraints
- Some sellers perform well in certain regions but poorly in others → indicates shipping partner inefficiency

### How do payment types, freight costs, and product categories affect order value?

In [0]:
from pyspark.sql.functions import avg, count, round as spark_round

df = (
    spark.table("ecom.gold.fact_orders")
    .join(spark.table("ecom.gold.fact_payments"), "order_id")
    .join(spark.table("ecom.gold.dim_products"), "product_id", "left")
)

df.groupBy("payment_type", "product_category_name_english") \
  .agg(
      spark_round(avg("price"), 2).alias("avg_order_value"),
      spark_round(avg("freight_value"), 2).alias("avg_freight_cost"),
      count("*").alias("total_orders")
  ) \
  .orderBy("avg_order_value", ascending=False) \
  .show()

+------------+-----------------------------+---------------+----------------+------------+
|payment_type|product_category_name_english|avg_order_value|avg_freight_cost|total_orders|
+------------+-----------------------------+---------------+----------------+------------+
|     voucher|                    computers|        1180.96|           44.18|          12|
| credit_card|                    computers|        1116.24|           45.92|         174|
|      boleto|                    computers|         1012.2|           60.05|          34|
|     voucher|         small_appliances_...|          750.0|            68.4|           3|
| credit_card|         small_appliances_...|         648.43|           36.28|          65|
| credit_card|            home_appliances_2|         554.98|           47.63|         186|
|      boleto|         small_appliances_...|         502.23|           34.81|           9|
|  debit_card|              fixed_telephony|         421.08|           24.22|           3|

### Final Takeaway

Order value is not driven by a single factor — it is the result of payment flexibility, shipping economics, and intrinsic product pricing working together.
Optimizing these three levers can significantly increase revenue and customer satisfaction.

### Which product categories generate high revenue but poor customer experience?

In [0]:
%sql
SELECT 
    p.product_category_name_english AS product_category,
    SUM(f.price) AS total_revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review_score,
    COUNT(*) AS total_orders
FROM ecom.gold.fact_orders f
JOIN ecom.gold.dim_products p ON f.product_id = p.product_id
GROUP BY p.product_category_name_english
HAVING SUM(f.price) > 500000
ORDER BY total_revenue DESC;

product_category,total_revenue,avg_review_score,total_orders
health_beauty,1259744.7399999667,4.14,9686
watches_gifts,1205256.4399999988,4.02,5997
bed_bath_table,1045390.5200000722,3.9,11202
sports_leisure,990729.7700000419,4.11,8667
computers_accessories,914932.3600000361,3.94,7854
furniture_decor,731629.2200000426,3.91,8364
cool_stuff,635820.5400000003,4.15,3800
housewares,634082.3600000212,4.06,6978
auto,593095.1600000112,4.07,4240


### Final Takeaway

- Categories like Electronics and Furniture are revenue leaders but experience laggards.
- Improving logistics and post-purchase support in these areas can significantly boost:

- Customer satisfaction 
- Repeat purchases 
- Brand trust 